# MTI850 – Analytiques des données massives
## Laboratoire Pratique : Prise en main d'Apache Spark & PySpark DataFrames

Ce notebook regroupe et adapte **les deux tutoriels officiels** demandés par le professeur Alessandro Koerich :
1. **[Spark Quick Start](https://spark.apache.org/docs/latest/quick-start.html)** : Les bases du moteur Spark, l'évaluation paresseuse (*Lazy Evaluation*), transformations vs actions, et la mise en cache (*caching*).
2. **[PySpark DataFrame Quickstart](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html)** : L'API DataFrame moderne (création, exploration, manipulation, regroupements, Spark SQL, et formats de stockage Big Data comme Parquet).

---
### Comment exécuter ce notebook ?
* Sélectionnez chaque cellule et appuyez sur **`Maj + Entrée`** (Shift + Enter).
* L'indicateur `In [*]` à gauche signifie que Spark est en train de calculer. Attendez qu'il devienne un numéro `In [1]` avant de continuer.


---
## Étape 0 : Initialisation de Spark avec `findspark` et `SparkSession`

Dans cette première étape, nous localisons les binaires de Spark avec `findspark`, puis nous instancions une **`SparkSession`**. La `SparkSession` est le point d'entrée unique de toute application Spark moderne.


In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, count, avg, max, min, sum, when
from datetime import datetime, date
import pandas as pd

# Création de la SparkSession en mode local
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("MTI850-Tutoriel-Pratique") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print("Spark version :", spark.version)
print("SparkSession initialisée avec succès !")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/12 20:46:27 WARN Utils: Your hostname, localhost, resolves to a loopback address: 127.0.0.1; using 172.21.0.2 instead (on interface eth0)
26/09/12 20:46:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/spark/python/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/12 20:46:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 4.2.0
SparkSession initialisée avec succès !


---
## PARTIE 1 : Spark Quick Start (Concepts Fondamentaux)

Cette section couvre les concepts clés du premier tutoriel officiel :
* La lecture d'un jeu de données
* La distinction cruciale entre **Transformations** et **Actions**
* La mise en cache mémoire avec `.cache()`


### 1.1 Lecture de données textuelles
Nous créons d'abord un jeu de données textuel d'exemple pour illustrer le traitement de logs ou de documents bruts.


In [2]:
sample_text = [
    "Apache Spark est un moteur d analyse unifie pour les donnees massives",
    "Spark offre des API de haut niveau en Java Scala Python et R",
    "Le moteur optimise de Spark supporte les graphes d execution generaux",
    "Spark fournit Spark SQL pour les donnees structurees",
    "Spark fournit MLlib pour l apprentissage automatique",
    "Spark fournit GraphX pour le calcul sur les graphes",
    "Spark Streaming permet le traitement de flux en temps reel",
    "Hadoop HDFS assure le stockage distribue et fiable des donnees",
    "Spark fonctionne tres bien avec Hadoop et HDFS",
    "L evaluation paresseuse permet a Spark d optimiser le plan d execution"
]

# Création d'un DataFrame de texte avec une seule colonne nommée "value"
text_df = spark.createDataFrame([(line,) for line in sample_text], ["value"])
text_df.show(5, truncate=False)


[Stage 1:=======================================>                   (2 + 1) / 3]

+---------------------------------------------------------------------+
|value                                                                |
+---------------------------------------------------------------------+
|Apache Spark est un moteur d analyse unifie pour les donnees massives|
|Spark offre des API de haut niveau en Java Scala Python et R         |
|Le moteur optimise de Spark supporte les graphes d execution generaux|
|Spark fournit Spark SQL pour les donnees structurees                 |
|Spark fournit MLlib pour l apprentissage automatique                 |
+---------------------------------------------------------------------+
only showing top 5 rows


### 1.2 Actions de base : `.count()` et `.first()`
Une **Action** force Spark à exécuter le calcul et renvoie un résultat au Driver (votre notebook).


In [3]:
# Compter le nombre de lignes (Action)
total_lines = text_df.count()
print(f"Nombre total de lignes : {total_lines}")

# Obtenir la première ligne (Action)
first_line = text_df.first()
print(f"Première ligne : {first_line['value']}")


Nombre total de lignes : 10
Première ligne : Apache Spark est un moteur d analyse unifie pour les donnees massives


### 1.3 Transformation : Filtrage avec `.filter()`
Une **Transformation** ne calcule rien immédiatement (*Lazy Evaluation*) : elle produit un nouveau DataFrame avec une règle appliquée.


In [4]:
# Filtrer toutes les lignes qui contiennent le mot 'Spark'
lines_with_spark = text_df.filter(text_df.value.contains("Spark"))

# Le calcul effectif n'a lieu que lorsqu'on déclenche une Action comme .count() ou .show()
print(f"Lignes contenant 'Spark' : {lines_with_spark.count()}")
lines_with_spark.show(truncate=False)


Lignes contenant 'Spark' : 9
+----------------------------------------------------------------------+
|value                                                                 |
+----------------------------------------------------------------------+
|Apache Spark est un moteur d analyse unifie pour les donnees massives |
|Spark offre des API de haut niveau en Java Scala Python et R          |
|Le moteur optimise de Spark supporte les graphes d execution generaux |
|Spark fournit Spark SQL pour les donnees structurees                  |
|Spark fournit MLlib pour l apprentissage automatique                  |
|Spark fournit GraphX pour le calcul sur les graphes                   |
|Spark Streaming permet le traitement de flux en temps reel            |
|Spark fonctionne tres bien avec Hadoop et HDFS                        |
|L evaluation paresseuse permet a Spark d optimiser le plan d execution|
+----------------------------------------------------------------------+



### 1.4 Exemple classique : Le Word Count (Comptage de mots)
Voici comment compter la fréquence des mots avec les transformations et actions de Spark :


In [5]:
from pyspark.sql.functions import split, explode, lower

# 1. Découper chaque ligne en mots (split)
# 2. Transformer chaque élément du tableau de mots en une ligne distincte (explode)
# 3. Mettre en minuscules et compter les occurrences
words_df = text_df.select(explode(split(lower(col("value")), " ")).alias("word")) \
    .filter(col("word") != "")

# Regroupement et comptage des 10 mots les plus fréquents
word_counts = words_df.groupBy("word").count().orderBy(col("count").desc())
word_counts.show(10)


[Stage 11:>                                                         (0 + 4) / 4]

+-------+-----+
|   word|count|
+-------+-----+
|  spark|   10|
|     le|    5|
|    les|    4|
|   pour|    4|
|      d|    4|
|donnees|    3|
|     de|    3|
|     et|    3|
|fournit|    3|
|graphes|    2|
+-------+-----+
only showing top 10 rows


### 1.5 La Mise en Cache (`.cache()`)
Dans le Big Data, recalculer un DataFrame à partir de zéro peut être très coûteux. Si un DataFrame doit être réutilisé plusieurs fois, on le place en cache dans la mémoire vive (RAM) du cluster grâce à `.cache()`.


In [6]:
import time

# Mettons en cache le DataFrame filtré
lines_with_spark.cache()

# Premier appel : Spark calcule et place les données en mémoire vive
t0 = time.time()
print("Premier count :", lines_with_spark.count())
print(f"Temps 1er appel : {(time.time() - t0)*1000:.2f} ms")

# Deuxième appel : Spark lit directement depuis la RAM !
t1 = time.time()
print("Deuxième count :", lines_with_spark.count())
print(f"Temps 2e appel (depuis le cache) : {(time.time() - t1)*1000:.2f} ms")


Premier count : 9
Temps 1er appel : 1081.64 ms
Deuxième count : 9
Temps 2e appel (depuis le cache) : 285.14 ms


---
## PARTIE 2 : PySpark DataFrame Quickstart (L'API Moderne)

Le DataFrame PySpark est une structure de données tabulaire distribuée, optimisée par le moteur **Catalyst Optimizer**. C'est le cœur du cours MTI850.


### 2.1 Création de DataFrames
Il existe plusieurs manières de créer un DataFrame Spark :
1. À partir de listes de tuples ou dictionnaires
2. À partir d'un DataFrame **pandas**
3. À partir de fichiers externes (CSV, JSON, Parquet)


In [7]:
# Création à partir d'une liste de tuples avec schéma explicite
clients_data = [
    (1, "Alice Martin", 28, "Montréal", 75000.0, date(2023, 1, 15)),
    (2, "Bob Tremblay", 42, "Québec", 92000.0, date(2022, 5, 20)),
    (3, "Charlie Roy", 35, "Laval", 68000.0, date(2023, 8, 10)),
    (4, "David Gagnon", 28, "Montréal", 81000.0, date(2021, 11, 5)),
    (5, "Emma Bouchard", 31, "Gatineau", 73000.0, date(2024, 2, 1)),
    (6, "François Côté", 45, "Montréal", 115000.0, date(2020, 3, 12))
]

colonnes = ["id", "nom", "age", "ville", "salaire", "date_embauche"]
df = spark.createDataFrame(clients_data, schema=colonnes)

print("DataFrame créé avec succès :")
df.show()


DataFrame créé avec succès :
+---+-------------+---+--------+--------+-------------+
| id|          nom|age|   ville| salaire|date_embauche|
+---+-------------+---+--------+--------+-------------+
|  1| Alice Martin| 28|Montréal| 75000.0|   2023-01-15|
|  2| Bob Tremblay| 42|  Québec| 92000.0|   2022-05-20|
|  3|  Charlie Roy| 35|   Laval| 68000.0|   2023-08-10|
|  4| David Gagnon| 28|Montréal| 81000.0|   2021-11-05|
|  5|Emma Bouchard| 31|Gatineau| 73000.0|   2024-02-01|
|  6|François Côté| 45|Montréal|115000.0|   2020-03-12|
+---+-------------+---+--------+--------+-------------+



### 2.2 Inspection du Schéma et Statistiques Descriptives
Spark infère automatiquement les types de données (entiers, chaînes, dates, décimaux).


In [8]:
# Afficher le schéma technique (colonnes et types)
df.printSchema()

# Afficher les statistiques de base (moyenne, écart-type, min, max)
df.describe(["age", "salaire"]).show()


root
 |-- id: long (nullable = true)
 |-- nom: string (nullable = true)
 |-- age: long (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: double (nullable = true)
 |-- date_embauche: date (nullable = true)



[Stage 23:=============================>                            (2 + 2) / 4]

+-------+------------------+------------------+
|summary|               age|           salaire|
+-------+------------------+------------------+
|  count|                 6|                 6|
|   mean|34.833333333333336|           84000.0|
| stddev|  7.25028735062733|17274.258305351348|
|    min|                28|           68000.0|
|    max|                45|          115000.0|
+-------+------------------+------------------+



### 2.3 Sélection, Ajout et Renommage de Colonnes
* `.select()` : choisir ou projeter des colonnes.
* `.withColumn()` : calculer ou ajouter une nouvelle colonne.
* `.withColumnRenamed()` : renommer une colonne.
* `.drop()` : supprimer une colonne.


In [9]:
# Sélectionner des colonnes et appliquer un calcul
df_modifie = df.select("nom", "ville", "salaire") \
    .withColumn("salaire_mensuel", (col("salaire") / 12).cast("decimal(10,2)")) \
    .withColumn("statut", when(col("salaire") >= 80000, "Senior").otherwise("Standard"))

df_modifie.show()


+-------------+--------+--------+---------------+--------+
|          nom|   ville| salaire|salaire_mensuel|  statut|
+-------------+--------+--------+---------------+--------+
| Alice Martin|Montréal| 75000.0|        6250.00|Standard|
| Bob Tremblay|  Québec| 92000.0|        7666.67|  Senior|
|  Charlie Roy|   Laval| 68000.0|        5666.67|Standard|
| David Gagnon|Montréal| 81000.0|        6750.00|  Senior|
|Emma Bouchard|Gatineau| 73000.0|        6083.33|Standard|
|François Côté|Montréal|115000.0|        9583.33|  Senior|
+-------------+--------+--------+---------------+--------+



### 2.4 Filtrage des données (`.filter()` / `.where()`)
Les conditions s'écrivent avec les opérateurs bitwise :
* `&` pour **ET**
* `|` pour **OU**
* `~` pour **NON**
*(Pensez à toujours encadrer chaque condition entre parenthèses).*


In [10]:
# Filtrer les employés de Montréal âgés de 30 ans et plus OU ayant un salaire > 80 000 $
df_filtre = df.filter(
    (col("ville") == "Montréal") & 
    ((col("age") >= 30) | (col("salaire") > 80000))
)

df_filtre.show()


+---+-------------+---+--------+--------+-------------+
| id|          nom|age|   ville| salaire|date_embauche|
+---+-------------+---+--------+--------+-------------+
|  4| David Gagnon| 28|Montréal| 81000.0|   2021-11-05|
|  6|François Côté| 45|Montréal|115000.0|   2020-03-12|
+---+-------------+---+--------+--------+-------------+



### 2.5 Regroupements et Agrégations (`.groupBy()` et `.agg()`)
Permet de calculer des métriques par catégorie (exactement comme le `GROUP BY` en SQL).


In [11]:
# Calculer par ville : nombre d'employés, salaire moyen, et salaire maximal
df_stats_ville = df.groupBy("ville").agg(
    count("id").alias("nb_employes"),
    avg("salaire").alias("salaire_moyen"),
    max("salaire").alias("salaire_max"),
    min("age").alias("age_min")
).orderBy(col("nb_employes").desc(), col("salaire_moyen").desc())

df_stats_ville.show()


+--------+-----------+-----------------+-----------+-------+
|   ville|nb_employes|    salaire_moyen|salaire_max|age_min|
+--------+-----------+-----------------+-----------+-------+
|Montréal|          3|90333.33333333333|   115000.0|     28|
|  Québec|          1|          92000.0|    92000.0|     42|
|Gatineau|          1|          73000.0|    73000.0|     31|
|   Laval|          1|          68000.0|    68000.0|     35|
+--------+-----------+-----------------+-----------+-------+



### 2.6 Exécution de Requêtes Spark SQL
Une des fonctionnalités les plus puissantes de Spark : vous pouvez exécuter directement des requêtes SQL ANSI standard sur vos DataFrames en créant une **vue temporaire**.


In [12]:
# Enregistrer le DataFrame comme une table SQL virtuelle temporaire
df.createOrReplaceTempView("employes")

# Écrire une requête SQL classique
sql_query = """
    SELECT 
        ville,
        COUNT(*) AS total_employes,
        ROUND(AVG(salaire), 2) AS salaire_moyen,
        MAX(salaire) AS salaire_max
    FROM employes
    WHERE age >= 30
    GROUP BY ville
    HAVING COUNT(*) >= 1
    ORDER BY salaire_moyen DESC
"""

resultat_sql = spark.sql(sql_query)
resultat_sql.show()


+--------+--------------+-------------+-----------+
|   ville|total_employes|salaire_moyen|salaire_max|
+--------+--------------+-------------+-----------+
|Montréal|             1|     115000.0|   115000.0|
|  Québec|             1|      92000.0|    92000.0|
|Gatineau|             1|      73000.0|    73000.0|
|   Laval|             1|      68000.0|    68000.0|
+--------+--------------+-------------+-----------+



### 2.7 Formats de Stockage Big Data : CSV vs Parquet
Dans un Data Lake ou un cluster Hadoop, le format **Apache Parquet** est le standard de l'industrie :
* Format **colonnaire** hautement compressé (utilise beaucoup moins d'espace disque).
* Permet le **Predicate Pushdown** : Spark ne lit sur disque que les colonnes et les lignes demandées par la requête, ce qui évite de parcourir tout le fichier.


In [13]:
import tempfile
import os

# Création d'un dossier temporaire de démonstration
temp_dir = tempfile.mkdtemp()
csv_path = os.path.join(temp_dir, "employes_csv")
parquet_path = os.path.join(temp_dir, "employes_parquet")

# 1. Sauvegarde au format CSV et Parquet
df.write.mode("overwrite").option("header", "true").csv(csv_path)
df.write.mode("overwrite").parquet(parquet_path)
print("Fichiers écrits avec succès !")

# 2. Relecture depuis le format Parquet
df_recharge = spark.read.parquet(parquet_path)
print("Données relues depuis Parquet :")
df_recharge.show(3)


Fichiers écrits avec succès !
Données relues depuis Parquet :
+---+-------------+---+--------+--------+-------------+
| id|          nom|age|   ville| salaire|date_embauche|
+---+-------------+---+--------+--------+-------------+
|  5|Emma Bouchard| 31|Gatineau| 73000.0|   2024-02-01|
|  6|François Côté| 45|Montréal|115000.0|   2020-03-12|
|  4| David Gagnon| 28|Montréal| 81000.0|   2021-11-05|
+---+-------------+---+--------+--------+-------------+
only showing top 3 rows


---
## PARTIE 3 : Bonnes pratiques et libération des ressources

Dans un environnement partagé ou sur une VM à ressources limitées :
1. Libérez le cache quand vous avez terminé (`unpersist()`).
2. Arrêtez proprement la `SparkSession` à la fin de vos travaux avec `spark.stop()`. Cela libère la mémoire JVM et les ports web.


In [14]:
# Libération du cache mémoire
lines_with_spark.unpersist()

# Arrêt propre de la SparkSession
# (Décommentez la ligne ci-dessous lorsque vous terminez votre session)
# spark.stop()
print("Session terminée proprement !")


Session terminée proprement !
